# 🏦 Bank Müştəri Churn Proqnozlaşdırılması (Random Forest)
**Məqsəd:** Müştərinin bankı tərk etmə ehtimalını (`churned`) proqnozlaşdırmaq - Binary Classification.

**Pipeline:** Aqreqasiya (müştəri səviyyəsi) → Preprocessing → Random Forest → Qiymətləndirmə → Feature Importance

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix

In [12]:
DATA_DIR = Path("/workspaces/Bank_Analysis/data/bank_data")

## 1. Data Ingestion & Feature Engineering (müştəri səviyyəsində aqreqasiya)
- **Demoqrafik məlumat:** `customers.csv` (hər müştəri 1 sətir, hədəf: `churned`).
- **Tranzaksiya davranışı:** `silver_transactions.csv` (Silverdə məzənnə `rate` və bayram `is_holiday` artıq birləşdirilib).
- Məzənnə `1 AZN = rate USD/EUR` formatındadır, ona görə AZN ekvivalenti = `amount / rate`.

In [13]:
# Tranzaksiyaları oxuyuruq və bütün məbləğləri AZN-ə çeviririk
tx = pd.read_csv(DATA_DIR / "silver_transactions.csv", low_memory=False)
tx["rate"] = tx["rate"].fillna(tx.groupby("currency")["rate"].transform("median"))
tx["rate"] = np.where(tx["currency"].str.lower() == "azn", 1.0, tx["rate"])
tx["amount_azn"] = tx["amount"] / tx["rate"]

In [14]:
# Hər müştəri üzrə xülasə (cəm, orta, say) çıxarırıq
tx_agg = tx.groupby("customer_id").agg(
    total_trans_count=("transaction_id", "count"),
    total_trans_amount=("amount_azn", "sum"),
    avg_trans_amount=("amount_azn", "mean"),
    holiday_trans_ratio=("is_holiday", "mean")
).reset_index()

In [15]:
# Müştəri profil cədvəlini oxuyub tranzaksiya xülasəsi ilə birləşdiririk
customers = pd.read_csv(DATA_DIR / "customers.csv")
customers["customer_id"] = customers["customer_id"].str.lower()
df = customers.drop(columns=["branch"], errors="ignore").merge(tx_agg, on="customer_id", how="left")

In [16]:
# Tranzaksiyası olmayan müştərilərin boş yerlərinə 0 yazırıq
trans_cols = ["total_trans_count", "total_trans_amount", "avg_trans_amount", "holiday_trans_ratio"]
df[trans_cols] = df[trans_cols].fillna(0)

# Yekun cədvəlin ölçüsü və Churn (itki) nisbəti
print(f"Cədvəl ölçüsü (Müştəri sayı, Sütun sayı): {df.shape}")
print(f"Bankı tərk edən müştəri nisbəti: {df['churned'].mean():.1%}")
df.head()

Cədvəl ölçüsü (Müştəri sayı, Sütun sayı): (5000, 16)
Bankı tərk edən müştəri nisbəti: 20.2%


,customer_id,age,gender,city,tenure_years,credit_score,balance,salary,products_count,has_card,is_active,churned,total_trans_count,total_trans_amount,avg_trans_amount,holiday_trans_ratio
0,c00001,39,Qadın,Bakı,2,654,3156.63,908.98,2,1,0,0,3.0,105.340000,35.113333,0.000000
1,c00002,47,Kişi,Sumqayıt,7,779,1327.10,807.53,2,1,1,0,17.0,2821.830000,165.990000,0.176471
2,c00003,18,Qadın,Gəncə,0,675,7973.01,1564.52,3,1,1,0,24.0,2401.054900,100.043954,0.083333
3,c00004,41,Kişi,Bakı,3,726,1338.89,1422.36,1,1,1,0,24.0,2241.884453,93.411852,0.208333
4,c00005,38,Kişi,Bakı,2,703,26360.17,2581.86,1,0,1,0,22.0,2150.131987,97.733272,0.136364


## 2. Data Preprocessing & Preparation
- Əskik dəyərlər: rəqəmsal → median, kateqorial → ən çox rast gələn (yalnız **train** üzərində öyrənilir → data leakage yoxdur).
- `gender`, `city` → One-Hot Encoding; rəqəmsal əlamətlər → `StandardScaler`.
- Stratified 80/20 split (`stratify=y`).

In [17]:
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

# Sütunların təyini
CAT_COLS, TARGET = ["gender", "city"], "churned"
NUM_COLS = [c for c in df.columns if c not in CAT_COLS + [TARGET, "customer_id"]]

In [18]:
# Train və Test-ə bölünmə (X və y-i ayrı dəyişən etmədən birbaşa bölürük)
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=[TARGET, "customer_id"]), 
    df[TARGET], 
    test_size=0.2, 
    stratify=df[TARGET], 
    random_state=42
)

In [19]:
# Yığcam Preprocessor (make_ pipeline və make_column_transformer ilə)
preprocessor = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM_COLS),
    (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), CAT_COLS)
)

print(f"Train: {X_train.shape} / Test: {X_test.shape}")
print(f"Churn payı - Train: {y_train.mean():.1%}, Test: {y_test.mean():.1%}")

Train: (4000, 14) / Test: (1000, 14)
Churn payı - Train: 20.2%, Test: 20.2%


## 3. Model Training (Random Forest)
`class_weight='balanced'` azlıq sinfinə (churn) daha çox çəki verir. Hiperparametrlər overfitting-i azaltmaq üçün mühafizəkar seçilib.

In [20]:
from sklearn.pipeline import make_pipeline

# Pipeline qurulur və öyrədilir
model = make_pipeline(
    preprocessor,
    RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=5,
        class_weight="balanced", random_state=42, n_jobs=-1
    )
)
model.fit(X_train, y_train)

# Test datası üzrə proqnozlar alınır
y_proba = model.predict_proba(X_test)[:, 1]  
y_pred = model.predict(X_test)               # Qəti qərar (0 və ya 1)

0.06s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
